In [1]:
from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

spark.version

'4.1.2'

In [4]:
spark.sql("use spark_catalog")
spark.sql("use schema northwind_db")

DataFrame[]

In [6]:
df_orders = spark.table("orders")
df_orders.createOrReplaceTempView("tmp_orders")
df_orders.printSchema()

root
 |-- OrderID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- EmployeeID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- RequiredDate: date (nullable = true)
 |-- ShippedDate: string (nullable = true)
 |-- ShipVia: string (nullable = true)
 |-- Freight: decimal(10,2) (nullable = true)
 |-- ShipName: string (nullable = true)
 |-- ShipAddress: string (nullable = true)
 |-- ShipCity: string (nullable = true)
 |-- ShipRegion: string (nullable = true)
 |-- ShipPostalCode: string (nullable = true)
 |-- ShipCountry: string (nullable = true)



In [7]:
from pyspark.sql.functions import current_date, current_timestamp, date_sub, date_add, to_date, lit, datediff, months_between, bround

df_orders2 = (df_orders.withColumn("today", current_date())
             .withColumn("now", current_timestamp())
             .withColumn("date_sub", date_sub("today", 5))
             .withColumn("date_add", date_add("today", 5))
             .withColumn("start", to_date(lit("2026-09-07")))
             .withColumn("days", datediff("today", "start"))
             .withColumn("months", bround(months_between("today", "start")))
             .select("start", "today", "days", "months","now", "date_sub", "date_add")
             )
display(df_orders2.limit(3))             

DataFrame[start: date, today: date, days: int, months: double, now: timestamp, date_sub: date, date_add: date]

In [8]:
spark.sql("""
select
    start,
    datediff(today, start) days,
    bround(months_between(today, start)) as months, 
    date_sub(today, 5) as date_sub,
    date_add(today, 5) as date_add,
    today,
    now
from
(
    select 
        to_date("2026-09-07") as start,
        current_date() as today,
        current_timestamp() as now
    from tmp_orders 
    limit 3
)
""")

DataFrame[start: date, days: int, months: double, date_sub: date, date_add: date, today: date, now: timestamp]

In [9]:
from pyspark.sql.functions import to_date, lit, to_timestamp, cast, col

df = spark.range(1)
date_fromat01 = "yyyy-MM-dd"
date_fromat02 = "yyyy-dd-MM"

df = (df.withColumn("date01", to_date(lit("2026-12-11"), date_fromat01))
      .withColumn("date02", to_date(lit("2026-13-11"), date_fromat02))
      .withColumn("date03", to_timestamp(lit("2026-13-11"), date_fromat02))
      )

display(df)

DataFrame[id: bigint, date01: date, date02: date, date03: timestamp]

In [11]:
spark.sql("""
with step1 as (
    select 
        to_date("2026-13-11", "yyyy-dd-MM") as date01,
        cast(to_date("2026-13-11", "yyyy-dd-MM") as timestamp) timestp
)
select 
    *,
    cast(timestp as date),
    to_date(timestp),
    try_cast(timestp as date)
from step1
""")

DataFrame[date01: date, timestp: timestamp, timestp: date, to_date(timestp): date, timestp: date]

# unix time

### ⏱️ Что такое UNIX-время (Unix Time / Epoch Time)?

**UNIX-время** — это система измерения времени, принятая в программировании и базах данных. Она измеряет время как **общее количество секунд, прошедших с полуночи (00:00:00 UTC) 1 января 1970 года**. Этот момент называют «Эпохой Unix» (Unix Epoch).

#### 📌 Ключевые особенности, важные для дата-инженерии:

1. **Независимость от часовых поясов (UTC):** 
   Unix-время всегда считается по всемирному координированному времени (UTC). Независимо от того, где находится сервер (в Нью-Йорке, Лондоне или Токио), в один и тот же момент времени значение Unix Timestamp будет одинаковым на всей планете. Часовые пояса применяются только в момент превращения этих секунд в понятную человеку строку (например, при вызове `from_unixtime()`).

2. **Проблема секунд и миллисекунд:**
   * В классическом понимании (и в стандартных функциях Spark) Unix-время измеряется в **секундах** (10-значное число, например, `1700000000`).
   * Многие современные системы (Kafka, JavaScript, логи приложений) отдают его в **миллисекундах** (13-значное число, например, `1700000000000`). 
   * *Перед конвертацией в формат даты в Spark миллисекунды обязательно нужно делить на 1000.*

3. **Простота для вычислений:**
   Поскольку Unix-время — это обычное целое число (`Long`), компьютеру с ним работать гораздо легче, чем с текстовыми строками вида `"2026-10-30 15:45:00"`. Сравнение дат, фильтрация по диапазону и сортировка на больших объемах данных (Big Data) через Unix Time происходят максимально быстро.

4. **«Проблема 2038 года» (Y2K38):**
   В старых 32-битных системах максимальное число секунд, которое можно сохранить, ограничено значением `2 147 483 647`. Это число будет достигнуто **19 января 2038 года в 03:14:07 UTC**, после чего счетчик «переполнится» и уйдет в минус (в 1901 год). В Apache Spark эта проблема отсутствует, так как для Unix-времени по умолчанию используется 64-битный тип (`LongType`), запаса которого хватит на миллиарды лет.


In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_unixtime, to_timestamp, unix_timestamp, current_timestamp

# Создаем тестовый DataFrame
data = [
    (1700000000, 1700000000000, "2023-11-14 22:13:20"),
    (1710000000, 1710000000000, "2024-03-09 16:00:00")
]
columns = ["ts_seconds", "ts_milliseconds", "date_string"]
df = spark.createDataFrame(data, columns)

# -------------------------------------------------------------------------
# 1. Конвертация из Unix Time (СЕКУНДЫ) в разные форматы
# -------------------------------------------------------------------------
df_converted = df.withColumn(
    "from_sec_to_string", 
    from_unixtime(col("ts_seconds"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "from_sec_to_timestamp", 
    col("ts_seconds").cast("timestamp") # Самый быстрый способ получить TimestampType
)

# -------------------------------------------------------------------------
# 2. Конвертация из Unix Time (МИЛЛИСЕКУНДЫ)
# -------------------------------------------------------------------------
df_converted = df_converted.withColumn(
    "from_ms_to_timestamp", 
    (col("ts_milliseconds") / 1000).cast("timestamp") # Обязательно делим на 1000
)

# -------------------------------------------------------------------------
# 3. Обратная конвертация: Из строки/Timestamp в Unix Time (секунды)
# -------------------------------------------------------------------------
df_converted = df_converted.withColumn(
    "string_to_unix", 
    unix_timestamp(col("date_string"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "current_time_unix",
    unix_timestamp(current_timestamp()) # Текущее системное Unix-время
)

# Вывод результатов
df_converted.select(
    "ts_seconds", "from_sec_to_string", "from_ms_to_timestamp", "string_to_unix"
).show(truncate=False)



NameError: name 'spark' is not defined